# Model M5: Unified Audit Priority Aggregator (Google Colab Notebook)
**SIH26102 MPLADS Audit Intelligence Platform**

Tests Model M5 (Composite Weight Matrix, sum(w_i) = 1.00).

In [ ]:
# Self-contained Repository & Path Setup for Google Colab
import os, sys, glob, zipfile

# 1. Look for unified_model_engine.py in current dir or subdirs
if not os.path.exists("unified_model_engine.py"):
    subdirs = [d for d in os.listdir(".") if os.path.isdir(d) and os.path.exists(os.path.join(d, "unified_model_engine.py"))]
    if subdirs:
        os.chdir(subdirs[0])
    else:
        zip_files = glob.glob("*.zip") + glob.glob("../*.zip")
        if zip_files:
            print(f"Extracting uploaded archive: {zip_files[0]}...")
            with zipfile.ZipFile(zip_files[0], 'r') as zip_ref:
                zip_ref.extractall(".")
            subdirs = [d for d in os.listdir(".") if os.path.isdir(d) and os.path.exists(os.path.join(d, "unified_model_engine.py"))]
            if subdirs:
                os.chdir(subdirs[0])
        else:
            !git clone https://github.com/Zaid5671/CodeBlooded.git
            if os.path.exists("CodeBlooded"):
                os.chdir("CodeBlooded")

sys.path.insert(0, os.getcwd())
os.environ["PYTHONPATH"] = os.getcwd()
print("Working Directory:", os.getcwd())
print("Engine Available:", os.path.exists("unified_model_engine.py"))
!pip install -q pandas numpy scikit-learn pytest

In [ ]:
# Run Model M5 Audit Priority Aggregator Directly
import json
import pandas as pd
from data_pipeline.data_loader import load_sanctioned_works
from feature_engineering.preprocessing import preprocess_sanctioned_works
from feature_engineering.category_classifier import apply_category_classification
from ml.model_1_cost_anomaly.overrun_rules import evaluate_cost_overrun
from ml.model_1_cost_anomaly.peer_analysis import evaluate_peer_iqr
from ml.model_1_cost_anomaly.isolation_forest import train_and_score_isolation_forest
from ml.model_5_audit_priority.misuse_priority import run_audit_priority_aggregation

print("=== TESTING MODEL M5 UNIFIED AUDIT PRIORITY AGGREGATOR ===")
df_raw = load_sanctioned_works()
df_prep = preprocess_sanctioned_works(df_raw)
df_cat = apply_category_classification(df_prep)
df_s1 = evaluate_cost_overrun(df_cat)
df_s2 = evaluate_peer_iqr(df_s1)
df_m1, imputer, iso_model = train_and_score_isolation_forest(df_s2)

df_priority, priority_summary = run_audit_priority_aggregation(df_scored=df_m1, df_delay=df_m1, df_compliance=df_m1)

print(f"Total Works Prioritized: {len(df_priority):,}")
print(f"Priority Summary:\n{json.dumps(priority_summary, indent=2)}")

# Display Top 5 Scored Priority Works
df_priority.sort_values(by='display_score', ascending=False)[['clean_work_id', 'state', 'sanction_amount', 'display_score', 'audit_priority']].head()